# DINOv2 v5 推理专用 — 3 seed 集成 + spec 专家融合 (不训练)

### 用途

加载 3 个 seed 成员 checkpoint (v5s1/s2/s3: `best_model_s42.pt` / `best_model_s142.pt` / `best_model_s242.pt`), 跳过训练直接做 gold 验证 + test 推理, rank-mean 融合后产出 `submission.csv`。

**spec 专家已归档 (2026-08-15)**: 离线扫描 `scripts/fusion_scan_spec.py` 确认 4 类替换为负收益 (-0.0023), 逐类 oracle 上限也仅 +0.0011 (< +0.003 阈值) → 融合端禁用。即使扫描到 `best_model_spec.pt` 也只打印诊断 AUC, 提交产物 = 纯 3 seed rank-mean, 与旧版行为一致。

### 运行前必做

1. **上传 3 个 checkpoint 为 Kaggle Dataset** (私有即可): 本地目录 `results/v5s{1,2,3}/checkpoints/` 里的 `best_model_s*.pt` (每个 ~132MB) 打包为 Dataset 根目录, 挂载到本 notebook; spec 专家 `results/v5spec/checkpoints/best_model_spec.pt` 可同目录或另建数据集 (可跳过: 已归档不参与融合, 仅诊断)
2. 若挂载路径不是 `CFG['ckpt_input']` 的默认值, 改 cell 2 的 `ckpt_input` — 或不改, notebook 会自动扫描 `/kaggle/input` 全部数据集找 `best_model_s*.pt`
3. **加速器 T4x2** (test 缓存大小未知, 与训练会话同规格)
4. **无需挂载** v5 标签数据集、无需挂载 DINOv2 预训练权重 (checkpoint 已包含全部权重)

### 预期输出

- 各成员 gold AUC 应与各自训练会话一致 (s42≈0.8933 / s142≈0.8899 / s242≈0.8937)
- 纯 seed 融合 gold 宏 AUC ≈ 0.896 (58 研究本地融合实测 0.8959)
- spec 若被扫到, 仅打印其诊断 AUC, FUSED = BASE
- `submission.csv` = 最终融合预测, 可直接提交

### 运行时长

gold 解码 ~2 分钟 + test 解码 (随测试集大小) + 成员推理 (~1-2 分钟/成员), 不训练, 全程远低于 9h 限制。



## 1. 导入


In [ ]:
# ============================================================
# v4: Imports
# ============================================================
from __future__ import annotations
import gc, math, os, re, sys, time
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import pydicom
import cv2
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

IS_MAIN = True
print('Imports OK.')



## 2. 配置 — 与 v5 训练数据侧参数一致


In [ ]:
# ============================================================
# 推理专用配置 — 与 v5 训练配置的数据侧参数必须逐项一致
# (cell 08 会拿每个 checkpoint 内的 config 与这里交叉核对, 不一致立即报错)
# ============================================================

TARGET_COLUMNS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture',
]
N_CLASSES = len(TARGET_COLUMNS)

# ---- 6 Clinical Slots ----
SLOTS = [
    ("SAG_FLUID_FS",   "Sagittal", True,  True),
    ("COR_FLUID_FS",   "Coronal",  True,  True),
    ("AX_FLUID_FS",    "Axial",    True,  True),
    ("SAG_FLUID_NOFS", "Sagittal", True,  False),
    ("COR_T1",         "Coronal",  False, False),
    ("SAG_T1",         "Sagittal", False, False),
]
N_SLOT = len(SLOTS)

# ---- Anatomical Priors ----
SLOT_PRIORS = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}

# ---- Diagnostic-specific TTA pooling (与训练一致) ----
DIAG_POOL = {
    "Fracture": "max", "Contusion": "max",
    "Medial Meniscus": "max", "Lateral Meniscus": "max",
    "Baker's": "max",
    "ACL": "top2", "MCL": "top2",
    "Synovitis": "original_mean",
}

# ---- Jitter TTA 增广参数 (与训练一致) ----
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
AUG_SEED = 42

CFG = {
    # --- Paths ---
    'comp_input': '/kaggle/input/competitions/rsna-knee-abnormality-detection',
    # ★ 你上传 3 个 best_model_s*.pt 为 Kaggle Dataset 后的挂载路径
    #   若 slug 不同也没关系: cell 08 会自动扫描 /kaggle/input 全部数据集找 best_model_s*.pt
    'ckpt_input': '/kaggle/input/datasets/easoncyy/v5-seed-checkpoints',
    'dicom_subdir': 'train_series',
    'output_dir': '/kaggle/working',

    # --- Data (★ 必须与 v5 训练一致, 勿改) ---
    'image_size': 288,             # 288px@130mm = 0.451mm/px
    'crop_mm': 130.0,
    'cache_slices': 9,
    'group_size': 3,
    'center_pct': (0.2, 0.8),

    # --- Model (★ 必须与 v5 训练一致, 勿改) ---
    'dinov2_variant': 'vit_small_patch14_dinov2.lvd142m',
    'cls_dim': 384,
    'feature_dim': 1152,
    'slot_hidden': 256,
    'num_classes': 12,
    'unfreeze_layers': 6,

    # --- Inference ---
    'tta_jitter': True,            # jitter TTA (训练时开启, 推理必须一致)
    'hdr_threads': 8,
    'pix_threads': 4,
}

N_WINDOWS = CFG['cache_slices'] - CFG['group_size'] + 1  # 7

import random
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(0)

N_GPUS = torch.cuda.device_count()
DEVICE = torch.device('cuda' if N_GPUS > 0 else 'cpu')

if IS_MAIN:
    print(f'GPUs: {N_GPUS} | Device: {DEVICE}')
    print(f'--- v5 推理专用 (seed 集成 rank-mean 融合) ---')
    print(f'  288px/130mm | {N_WINDOWS} 窗口 TTA + jitter | 诊断池化')



## 3. Slot 匹配 + 侧性检测


In [ ]:
# ============================================================
# v4: Slot Matching + Laterality Detection + DICOM Header Annotation
# ============================================================

# ---- DICOM Header Annotation (Ref1: annotate_sequences) ----
_SEP = re.compile(r'[_\-.]')
_FATSAT_RX = re.compile(
    r'\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|'
    r'water excit|\btirm\b|\bsting\b|\bfatsup\b'
)
_T1_RX = re.compile(r'\bt1\b|\bt1w\b')
_T2_RX = re.compile(r'\bt2\b|\bt2w\b')
_PD_RX = re.compile(r'\bpd\b|\bpdw\b|proton|\bdp\b|dens')

FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}

_HDR_TAGS = [
    'SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence',
    'RepetitionTime', 'EchoTime', 'Laterality', 'ImageLaterality',
    'ImagePositionPatient', 'PixelSpacing',
]


def _tag_side(group):
    """从 DICOM Laterality 标签推断侧性。"""
    values = [str(x).strip().upper() for x in group.get('Laterality', pd.Series(dtype=object)).dropna()]
    if 'ImageLaterality' in group.columns:
        values += [str(x).strip().upper() for x in group['ImageLaterality'].dropna()]
    values = [x[0] for x in values if x and x[0] in ('L', 'R')]
    return values[0] if values else None


def _position_side(group, min_offset_mm=5.0):
    """从 ImagePositionPatient[0] 推断侧性：DICOM LPS 中 +x = 患者左侧。"""
    xs = []
    for raw in group.get('ImagePositionPatient', pd.Series(dtype=object)).dropna():
        try:
            xs.append(float(str(raw).split('|')[0]))
        except Exception:
            pass
    if not xs:
        return None
    median_x = float(np.median(xs))
    if abs(median_x) < min_offset_mm:
        return None
    return 'R' if median_x < 0 else 'L'


def detect_laterality(headers_df):
    """为每个 study 确定侧性（左/右），结合标签和几何位置。"""
    tagged, positioned = {}, {}
    for study_uid, group in headers_df.groupby('StudyInstanceUID'):
        tagged[study_uid] = _tag_side(group)
        positioned[study_uid] = _position_side(group)

    comparable = [s for s in tagged if tagged[s] and positioned[s]]
    agreement = float(np.mean([
        tagged[s] == positioned[s] for s in comparable
    ])) if comparable else np.nan

    use_position = bool(comparable) and np.isfinite(agreement) and agreement >= 0.85

    resolved = {
        uid: (tagged[uid] or (positioned[uid] if use_position else None))
        for uid in tagged
    }
    coverage = float(np.mean([v is not None for v in resolved.values()]))

    if IS_MAIN:
        print(f'Laterality: tag_coverage={len([v for v in tagged.values() if v])/max(len(tagged),1):.1%}, '
              f'agreement={agreement:.1%} on {len(comparable)} studies, '
              f'final_coverage={coverage:.1%}')
    return resolved


def annotate_sequences(df):
    """从 DICOM header 推断 Fluid/FatSat/Weight，作为 train_series.csv 的 fallback。"""
    df = df.copy()

    # Fat suppression detection
    desc = (df.get('SeriesDescription', '').fillna('') + ' ' +
            df.get('SequenceName', '').fillna(''))
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)

    scan_options = df.get('ScanOptions', '').fillna('').str.upper().str.split('|')
    option_fatsat = scan_options.apply(
        lambda tokens: any(t.strip() in FATSAT_OPTS for t in tokens))
    df['fatsat_detected'] = desc.str.contains(_FATSAT_RX) | option_fatsat

    # Weight detection
    tr = pd.to_numeric(df.get('RepetitionTime', np.nan), errors='coerce')
    te = pd.to_numeric(df.get('EchoTime', np.nan), errors='coerce')
    named_t1 = desc.str.contains(_T1_RX)
    named_t2 = desc.str.contains(_T2_RX)
    named_pd = desc.str.contains(_PD_RX)

    df['weight'] = np.where(
        named_t1 & ~named_t2 & ~named_pd, 'T1',
        np.where(named_t2 & ~named_pd, 'T2',
                 np.where(named_pd, 'PD',
                          np.where(tr < 800, 'T1',
                                   np.where(te > 60, 'T2',
                                            np.where(tr >= 800, 'PD', 'UNK'))))))
    df['fluid_detected'] = df['weight'].isin(['PD', 'T2'])

    return df


# ---- Slot Matching ----
def match_slots_for_study(study_series_df):
    """为单个 study 的每个 slot 匹配最优 series。"""
    slots_found = {}
    for slot_name, plane, fluid, fatsat in SLOTS:
        candidates = study_series_df[
            (study_series_df['Anatomical_Plane'] == plane)
            & (study_series_df['Fluid_Sensitive'] == (1 if fluid else 0))
            & (study_series_df['Fat_Suppression'] == (1 if fatsat else 0))
        ]
        if len(candidates) == 0 and not fluid:
            candidates = study_series_df[
                (study_series_df['Anatomical_Plane'] == plane)
                & (study_series_df['Fluid_Sensitive'] == 0)
            ]
        if len(candidates) > 0:
            best = candidates.sort_values('n_slices', ascending=False).iloc[0]
            slots_found[slot_name] = {
                'series_uid': best['SeriesInstanceUID'],
                'dir': best['dir'],
                'n_slices': int(best['n_slices']),
                'plane': plane,
            }
        else:
            slots_found[slot_name] = None
    return slots_found


def build_study_slot_map(series_meta, dicom_root):
    """为所有 study 构建 slot→series 映射。"""
    df = series_meta.copy()
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    df['SeriesInstanceUID'] = df['SeriesInstanceUID'].astype(str)

    # 计算 DICOM 目录和切片数
    dirs, n_slices_list = [], []
    for _, row in df.iterrows():
        d = str(dicom_root / row['StudyInstanceUID'] / row['SeriesInstanceUID'])
        dirs.append(d)
        if os.path.isdir(d):
            files = [f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]
            n_dcm = len([f for f in files if f.endswith('.dcm')])
            if n_dcm == 0:
                n_dcm = len([f for f in files if not f.startswith('.')])
            n_slices_list.append(n_dcm)
        else:
            n_slices_list.append(0)
    df['dir'] = dirs
    df['n_slices'] = n_slices_list

    slot_map, study_series_map = {}, {}
    for study_uid, grp in df.groupby('StudyInstanceUID'):
        study_series_map[study_uid] = grp
        slot_map[study_uid] = match_slots_for_study(grp)

    # 统计
    slot_counts = {}
    for slots in slot_map.values():
        for name, sid in slots.items():
            slot_counts[name] = slot_counts.get(name, 0) + (1 if sid is not None else 0)

    if IS_MAIN:
        n_studies = len(slot_map)
        print(f'Slot map: {n_studies} studies')
        for name, count in slot_counts.items():
            print(f'  {name:<18s}: {count:5d}/{n_studies} ({count/n_studies*100:.0f}%)')

    return slot_map, study_series_map

print('Slot matching v4 ready.')



## 4. DICOM — 空间排序 + 物理裁剪 + 侧性归一化


In [ ]:
# ============================================================
# v4: DICOM I/O — 空间排序 + 物理裁剪 + 侧性归一化 + 并行读取
# ============================================================

# ---- 空间切片排序 (Ref2: dominant_axis) ----
PLANE_AXIS = {"Sagittal": 0, "Coronal": 1, "Axial": 2}

def _list_dcm_files(series_dir):
    """列出 DICOM 文件（不依赖 .dcm 扩展名，竞赛 test 集无后缀）。"""
    sd = Path(series_dir)
    if not sd.is_dir():
        return []
    all_files = sorted(f.name for f in sd.iterdir() if f.is_file())
    dcm = [f for f in all_files if f.endswith('.dcm')]
    return dcm if dcm else [f for f in all_files if not f.startswith('.')]

def spatially_sorted_files(series_dir, plane=None):
    """按 ImagePositionPatient 在切片法线方向上的投影排序。
    文件名排序的 Spearman 相关系数仅 0.009——完全随机。
    """
    series_dir = Path(series_dir)
    files = _list_dcm_files(series_dir)
    if not files:
        return []

    axis = PLANE_AXIS.get(plane, 2)
    rows = []
    for fname in files:
        try:
            ds = pydicom.dcmread(
                str(series_dir / fname), stop_before_pixels=True, force=True,
                specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            ipp = getattr(ds, 'ImagePositionPatient', None)
            instance = getattr(ds, 'InstanceNumber', None)
            if ipp is not None and len(ipp) >= 3:
                candidate = np.array(ipp[:3], dtype=np.float64)
                pos = float(candidate[axis]) if np.isfinite(candidate).all() else None
            else:
                pos = None
            inst_val = float(instance) if instance is not None else None
        except Exception:
            pos, inst_val = None, None
        rows.append((fname, pos, inst_val))

    positioned = [r for r in rows if r[1] is not None]
    threshold = max(2, int(0.8 * len(rows)))

    if len(positioned) >= threshold:
        # 主排序：通过平面坐标
        rows.sort(key=lambda r: (
            r[1] if r[1] is not None else 0.0,
            r[2] if r[2] is not None else float('inf'),
        ))
    elif sum(r[2] is not None for r in rows) >= threshold:
        rows.sort(key=lambda r: (
            r[2] if r[2] is not None else float('inf'),
        ))
    # else: 保持文件名顺序

    return [r[0] for r in rows]


# ---- 侧性归一化 ----
def normalise_laterality(image, plane, laterality):
    """右膝映射为左膝：冠/轴面水平翻转，矢面反转切片顺序。"""
    if laterality != 'R':
        return image
    # image: [N_slices, H, W] numpy
    if plane in ('Coronal', 'Axial'):
        return np.flip(image, axis=-1).copy()  # 水平翻转
    else:
        return np.flip(image, axis=0).copy()    # 反转切片顺序


# ---- 物理裁剪 ----
def physical_crop(volume, px, crop_mm=160.0):
    """基于 PixelSpacing 裁剪到固定物理 FOV，消除不同扫描仪的空间尺度差异。"""
    if px is None or not np.isfinite(px) or px <= 0:
        return volume
    desired = int(round(crop_mm / px))
    h, w = volume.shape[1], volume.shape[2]
    if not (16 < desired < min(h, w)):
        return volume
    cy, cx = h // 2, w // 2
    half = desired // 2
    return volume[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]


# ---- 读取单 series 为 volume ----
def read_series_volume(series_dir, plane=None, laterality=None,
                       image_size=224, crop_mm=160.0):
    """读取 DICOM 序列 → 空间排序 → 物理裁剪 → 侧性归一化 → 归一化 → 缩放。"""
    sorted_files = spatially_sorted_files(series_dir, plane)
    if not sorted_files:
        return None, None

    series_dir = Path(series_dir)
    slices_info = []
    px = None

    for fname in sorted_files:
        try:
            ds = pydicom.dcmread(str(series_dir / fname), force=True)
            img = ds.pixel_array.astype(np.float32)

            # Rescale
            slope = float(getattr(ds, 'RescaleSlope', 1) or 1)
            intercept = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            img = img * slope + intercept

            # PixelSpacing (取第一个有效值)
            if px is None:
                ps = getattr(ds, 'PixelSpacing', None)
                if ps is not None and len(ps) >= 1:
                    try:
                        px = float(ps[0])
                    except Exception:
                        pass

            slices_info.append(img)
        except Exception:
            slices_info.append(np.zeros((image_size, image_size), dtype=np.float32))

    if not slices_info:
        return None, None

    volume = np.stack(slices_info, axis=0)  # [N, H, W]

    # 物理裁剪
    volume = physical_crop(volume, px, crop_mm)

    # 侧性归一化
    volume = normalise_laterality(volume, plane, laterality)

    # 鲁棒归一化 (1st-99th percentile)
    v_low, v_high = np.percentile(volume, [1.0, 99.0])
    volume = np.clip(volume, v_low, v_high)
    denom = max(v_high - v_low, 1e-6)
    volume = (volume - v_low) / denom

    # 缩放到 target size
    resized = []
    for img in volume:
        r = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        resized.append(r)
    return np.stack(resized, axis=0).astype(np.float32), px


# ---- 缓存切片采样 ----
def sample_cache_slices(volume, n_cache=9, center_pct=(0.2, 0.8)):
    """从 volume 的 central 60% 区域均匀采样 n_cache 个切片。"""
    n_total = volume.shape[0]
    if n_total <= n_cache:
        indices = list(range(n_total))
        while len(indices) < n_cache:
            indices.append(indices[-1])
        return volume[np.array(indices)]

    low = int(center_pct[0] * (n_total - 1))
    high = int(center_pct[1] * (n_total - 1))
    if high <= low:
        low, high = 0, n_total - 1
    indices = np.unique(np.linspace(low, high, n_cache).astype(int))
    while len(indices) < n_cache:
        indices = np.append(indices, indices[-1])
    return volume[indices[:n_cache]]

print('DICOM I/O v4 ready.')



## 5. 模型 — SlotHead + MultiViewModel + 诊断池化


In [ ]:
# ============================================================
# v4: SlotHead + MultiViewModel — 支持诊断池化
# ============================================================

class SlotHead(nn.Module):
    """Per-diagnosis attention over MRI slots with anatomical priors."""

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

        prior = torch.zeros(n_out, n_slot)
        for target_name, slot_indices in SLOT_PRIORS.items():
            if target_name in TARGET_COLUMNS:
                prior[TARGET_COLUMNS.index(target_name), list(slot_indices)] = 0.55
        self.register_buffer("slot_prior", prior)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb                              # [B, S, H]
        attention = (
            torch.einsum("bsh,oh->bos", h, self.query)                # [B, n_out, S]
            / math.sqrt(self.hidden)
            + self.slot_prior.unsqueeze(0)
        )
        attention = attention.masked_fill(
            mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        context = self.drop(torch.einsum("bos,bsh->boh", attention, h))
        return (context * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class MultiViewModel(nn.Module):
    """DINOv2 + SlotHead for multi-view knee MRI."""

    def __init__(self, dinov2_model, n_slots=6, cls_dim=384,
                 n_classes=12, slot_hidden=256, dropout=0.2,
                 unfreeze_layers=6):
        super().__init__()
        self.n_slots = n_slots
        self.cls_dim = cls_dim
        self.feature_dim = cls_dim * 3
        self.unfreeze_layers = unfreeze_layers

        self.dinov2 = dinov2_model
        n_blocks = len(self.dinov2.blocks)
        if unfreeze_layers > 0:
            for p in self.dinov2.parameters():
                p.requires_grad = False
            unfreeze_start = max(0, n_blocks - unfreeze_layers)
            for block in self.dinov2.blocks[unfreeze_start:]:
                for p in block.parameters():
                    p.requires_grad = True
            if hasattr(self.dinov2, 'norm'):
                for p in self.dinov2.norm.parameters():
                    p.requires_grad = True

        self.head = SlotHead(
            dim=self.feature_dim, n_slot=n_slots, n_out=n_classes,
            hidden=slot_hidden, p=dropout)

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _extract_features(self, x_3ch):
        if self.unfreeze_layers > 0:
            features = self.dinov2.forward_features(x_3ch)
        else:
            with torch.no_grad():
                features = self.dinov2.forward_features(x_3ch)
        cls = features[:, 0, :]
        patches = features[:, 1:, :]
        mean_p = patches.mean(dim=1)
        k = max(1, patches.shape[1] // 8)
        focal = patches.topk(k, dim=1).values.mean(dim=1)
        return torch.cat([cls, mean_p, focal], dim=1)

    def forward(self, images, mask):
        """images: [B, S, 3, H, W] uint8 or [B*W, S, 3, H, W] for TTA"""
        B, S = images.shape[:2]
        x = images.reshape(B * S, 3, images.shape[-2], images.shape[-1])
        x = x.float().div_(255.0)
        x = (x - self.mean) / self.std
        features = self._extract_features(x)
        features = features.reshape(B, S, -1)
        return self.head(features, mask)

    def train(self, mode=True):
        super().train(mode)
        self.dinov2.eval()
        return self


# ---- ★ Jitter TTA 增广视图 (0.91 notebook augment() 移植) ----
def tta_jitter(imgs, seed=AUG_SEED):
    """每窗口生成一个确定性增广视图。

    几何（旋转 ±AUG_ROT_DEG° / 缩放 +[0, AUG_SCALE] / 平移 ±AUG_SHIFT）+
    强度 ±AUG_INTENSITY，border 填充（0.91 同款）。
    固定种子 → 同一批输入每次生成相同增广，验证/测试/提交全程可复现。
    输入 [..., 3, H, W] uint8 → 输出同形状同 dtype。
    """
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = (x.shape[0], x.device)
    gen = torch.Generator(device=dev).manual_seed(int(seed) % (2 ** 63 - 1))

    rot = (torch.rand(n, device=dev, generator=gen) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    sc = 1.0 + torch.rand(n, device=dev, generator=gen) * AUG_SCALE
    tx = (torch.rand(n, device=dev, generator=gen) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev, generator=gen) - 0.5) * 2 * AUG_SHIFT
    cos, sin = (torch.cos(rot) / sc, torch.sin(rot) / sc)

    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = (cos, -sin, tx)
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = (sin, cos, ty)

    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)

    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev, generator=gen) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)


def stack_views(logits_flat, B, W, n_orig):
    """TTA 视图分组: [V*B*W, C]（B-major：每研究 W 行连续，原始块在前）→ [B, V*W, C]。

    V=2（jitter 开启）时输出每研究 [前 W 行原始视图, 后 W 行增广视图]；
    n_orig=None（无 jitter）时即 [B, W, C]。
    ★ 不可用 reshape(B, -1, C) 直接切——行序是研究大循环，会跨研究串位。
    """
    if n_orig is None:
        return logits_flat.reshape(B, W, -1)
    return logits_flat.view(2, B, W, -1).permute(1, 0, 2, 3).reshape(B, 2 * W, -1)


# ---- ★ 诊断特异性 TTA 池化 ----
DIAG_POOL_IDX = {}
for target_name, mode in DIAG_POOL.items():
    if target_name in TARGET_COLUMNS:
        DIAG_POOL_IDX[TARGET_COLUMNS.index(target_name)] = mode


def diagnostic_pool(logits_views, pool_idx=None, n_orig=None):
    """对 [B, V, C] logits 应用诊断特异性池化。

    - max:           局部病灶保留最强信号窗口
    - top2:          ACL/MCL 取前2强窗口平均
    - mean:          弥漫性病变取全窗口平均（默认）
    - original_mean: 仅无 jitter 原始视图平均（Synovitis, 0.91 同款）

    jitter TTA 模式（n_orig 给定）: 前 n_orig 个视图为原始视图、其余为增广视图；
    先按窗口做视图平均（0.91 的 win_probs），再做 per-target 窗口池化。
    n_orig=None 时全部视图视为原始视图（original_mean ≡ mean，与旧版行为一致）。
    """
    if pool_idx is None:
        pool_idx = DIAG_POOL_IDX

    B, V, C = logits_views.shape
    if n_orig is not None:
        orig_probs = torch.sigmoid(logits_views[:, :n_orig])             # [B, W, C]
        probs = (orig_probs + torch.sigmoid(logits_views[:, n_orig:])) / 2  # 视图平均
    else:
        probs = torch.sigmoid(logits_views)
        orig_probs = probs

    result = probs.mean(dim=1)                          # [B, C] — 默认 mean

    for j, mode in pool_idx.items():
        x = probs[:, :, j]                             # [B, W]
        if mode == 'max':
            result[:, j] = x.max(dim=1).values
        elif mode == 'top2':
            result[:, j] = x.topk(min(2, x.shape[1]), dim=1).values.mean(dim=1)
        elif mode == 'original_mean':
            result[:, j] = orig_probs[:, :, j].mean(dim=1)

    return result  # [B, C]


if IS_MAIN:
    n_total = sum(p.numel() for p in SlotHead(1152, 6, 12).parameters())
    print(f'SlotHead params: {n_total/1e6:.3f}M')
    print(f'Diag pool targets: {list(DIAG_POOL_IDX.keys())}')
    print(f'Jitter TTA: {"ON" if CFG.get("tta_jitter", False) else "OFF"} '
          f'(rot ±{AUG_ROT_DEG:.0f}°, scale +{AUG_SCALE:.0%}, '
          f'shift ±{AUG_SHIFT:.0%}, intensity ±{AUG_INTENSITY:.0%})')
    print('Model v4 ready.')



## 6. 加载 gold 研究 (仅验证, 无需 v5 标签数据集)


In [ ]:
# ============================================================
# 加载 gold (58 全标注研究) — 仅用于验证, 推理 notebook 无需 v5 标签数据集
# ============================================================

comp_input = Path(CFG['comp_input'])

# ---- 竞赛元数据 ----
train_meta = pd.read_csv(comp_input / 'train.csv')
train_meta['StudyInstanceUID'] = train_meta['StudyInstanceUID'].astype(str)

# ---- 58 gold (12 类全标注) ----
label_cols_present = [c for c in TARGET_COLUMNS if c in train_meta.columns]
has_all_labels = train_meta[label_cols_present].notna().all(axis=1)
gold_df = train_meta[has_all_labels].copy()
gold_studies = sorted(gold_df['StudyInstanceUID'].unique())

gold_labels = gold_df[['StudyInstanceUID'] + label_cols_present].copy()
gold_labels = gold_labels.set_index('StudyInstanceUID')
for c in TARGET_COLUMNS:
    if c not in gold_labels.columns:
        gold_labels[c] = np.nan
gold_labels = gold_labels.apply(pd.to_numeric, errors='coerce')

n_pos_per_class = (gold_labels > 0).sum(axis=0)

# ---- train_series 元数据 (只保留 gold 研究, 供 slot 匹配) ----
series_meta = pd.read_csv(comp_input / 'train_series.csv')
series_meta['StudyInstanceUID'] = series_meta['StudyInstanceUID'].astype(str)
series_meta['SeriesInstanceUID'] = series_meta['SeriesInstanceUID'].astype(str)
gold_series_meta = series_meta[
    series_meta['StudyInstanceUID'].isin(set(gold_studies))].copy()

if IS_MAIN:
    print(f'Gold studies (all 12 labeled): {len(gold_studies)} (expect 58)')
    print(f'Gold positives per class: min={int(n_pos_per_class.min())}, '
          f'max={int(n_pos_per_class.max())}, mean={n_pos_per_class.mean():.1f}')
    print(f'Gold series meta rows: {len(gold_series_meta)}')



## 7. 构建 gold 缓存 (58 研究, ~2 分钟)


In [ ]:
# ============================================================
# 构建 gold 研究缓存 (仅 58 研究, 与训练期解码逻辑逐字一致)
# 训练期整个 train 缓存要 ~1h, 这里只解 58 个 gold → ~2 分钟
# ============================================================

dicom_root = Path(CFG['comp_input']) / CFG['dicom_subdir']
print(f'DICOM root: {dicom_root}')

# ---- slot 匹配 (仅 gold 研究) ----
slot_map, _ = build_study_slot_map(gold_series_meta, dicom_root)

needed_slot_map = {uid: slot_map[uid] for uid in gold_studies if uid in slot_map}
print(f'Gold studies with slot map: {len(needed_slot_map)}/{len(gold_studies)}')

# ---- 快速侧性检测 (训练期同款) ----
def _detect_laterality_fast(needed_slot_map):
    """为每个 study 快速检测侧性（只读每个 study 第一个有效 series 的 header）。"""
    laterality_map = {}
    for study_uid, study_slots in needed_slot_map.items():
        lat = None
        for slot_name, slot_info in study_slots.items():
            if slot_info is None:
                continue
            series_dir = Path(slot_info['dir']) if 'dir' in slot_info else None
            if series_dir is None or not series_dir.exists():
                continue
            dcm_files = _list_dcm_files(series_dir)
            if not dcm_files:
                continue
            try:
                ds = pydicom.dcmread(
                    str(series_dir / dcm_files[0]), stop_before_pixels=True, force=True,
                    specific_tags=['Laterality', 'ImageLaterality', 'ImagePositionPatient'])
                for tag_name in ['Laterality', 'ImageLaterality']:
                    val = getattr(ds, tag_name, None)
                    if val is not None:
                        val = str(val).strip().upper()
                        if val and val[0] in ('L', 'R'):
                            lat = val[0]
                            break
                if lat is not None:
                    break
                ipp = getattr(ds, 'ImagePositionPatient', None)
                if ipp is not None and len(ipp) >= 1:
                    try:
                        x = float(str(ipp[0]).split('\\')[0].split('|')[0])
                        if abs(x) >= 5.0:
                            lat = 'R' if x < 0 else 'L'
                            break
                    except Exception:
                        pass
            except Exception:
                continue
        laterality_map[study_uid] = lat
    return laterality_map

t_lat = time.time()
laterality_map = _detect_laterality_fast(needed_slot_map)
n_lat = sum(1 for v in laterality_map.values() if v is not None)
n_right = sum(1 for v in laterality_map.values() if v == 'R')
if IS_MAIN:
    print(f'Laterality detected: {n_lat}/{len(laterality_map)} studies '
          f'({n_lat/max(len(laterality_map),1)*100:.1f}%), '
          f'R={n_right}, L={n_lat-n_right}, ({time.time()-t_lat:.1f}s)')

# ---- 预分配缓存 ----
cache_shape = (len(gold_studies), N_SLOT, CFG['cache_slices'],
               CFG['image_size'], CFG['image_size'])

GOLD_CACHE = np.zeros(cache_shape, dtype=np.uint8)
GOLD_MASK = np.zeros((len(gold_studies), N_SLOT), dtype=np.float32)
gold_study_index = {uid: i for i, uid in enumerate(gold_studies)}

print(f'Gold cache: {cache_shape} = {GOLD_CACHE.nbytes / 1024**2:.1f} MB uint8')

# ---- 并行 DICOM 读取 (训练期同款 _read_slot_job) ----
def _read_slot_job(args):
    """单个 slot 的读取任务（用于 ThreadPoolExecutor）"""
    row_idx, slot_idx, slot_name, plane, slot_info, laterality = args
    if slot_info is None:
        return row_idx, slot_idx, None

    series_dir = Path(slot_info['dir']) if 'dir' in slot_info else None
    if series_dir is None or not series_dir.exists():
        return row_idx, slot_idx, None

    try:
        volume, px = read_series_volume(
            str(series_dir), plane=plane, laterality=laterality,
            image_size=CFG['image_size'], crop_mm=CFG['crop_mm'])
        if volume is None or volume.shape[0] < 3:
            return row_idx, slot_idx, None

        sampled = sample_cache_slices(
            volume, n_cache=CFG['cache_slices'], center_pct=CFG['center_pct'])
        sampled_uint8 = (sampled * 255).clip(0, 255).round().astype(np.uint8)
        return row_idx, slot_idx, sampled_uint8
    except Exception:
        return row_idx, slot_idx, None


t_cache = time.time()
jobs = []
for row_idx, study_uid in enumerate(gold_studies):
    study_slots = needed_slot_map.get(study_uid, {})
    lat = laterality_map.get(study_uid)
    for slot_idx, (slot_name, plane, fluid, fatsat) in enumerate(SLOTS):
        slot_info = study_slots.get(slot_name)
        if slot_info is not None:
            jobs.append((row_idx, slot_idx, slot_name, plane, slot_info, lat))

print(f'Decoding {len(jobs)} gold slot-series (parallel, {CFG["pix_threads"]} threads)...')
completed = 0
failed = 0
with ThreadPoolExecutor(max_workers=CFG['pix_threads']) as pool:
    for row_idx, slot_idx, result in pool.map(_read_slot_job, jobs):
        completed += 1
        if result is not None:
            GOLD_CACHE[row_idx, slot_idx] = result
            GOLD_MASK[row_idx, slot_idx] = 1.0
        else:
            failed += 1
        if completed % 100 == 0:
            print(f'  [{completed}/{len(jobs)}] {time.time()-t_cache:.0f}s', flush=True)

total_series = int(GOLD_MASK.sum())
print(f'\nGold cache built: {len(gold_studies)} studies, {total_series} series, '
      f'{GOLD_CACHE.nbytes / 1024**2:.1f} MB in {time.time()-t_cache:.0f}s')
print(f'  Failed reads: {failed}')
print(f'  ★ Physical crop: {CFG["crop_mm"]}mm | Laterality: {n_right}R/{n_lat-n_right}L | Threads: {CFG["pix_threads"]}')

gc.collect()



## 8. Seed 集成推理 + rank-mean 融合 + Submission


In [ ]:
# ============================================================
# ★ 集成推理 + 融合 (跳过训练, 纯推理)
#
# 成员: 3 个 seed (best_model_s{42,142,242}.pt)
#       + 可选 1 个 spec 专家 (best_model_spec.pt, 词表标签监督; 仅诊断, 不融合)
# 流程:
#   1. 找到全部成员 checkpoint (best_model_s*.pt)
#   2. 逐个与训练配置交叉核对 (数据侧参数不一致 → 立即报错)
#   3. gold 验证: 每成员 7 窗口 TTA + jitter + 诊断池化 → 逐类 AUC
#   4. test 推理: 每成员同款 TTA
#   5. 融合: 3 seed rank-mean (spec 成员已归档: 离线扫描 5 方案 ≤ +0.001,
#      当前 4 类替换为负 -0.0023 → 不参与融合, 只打印诊断 AUC)
#   6. submission.csv
# ============================================================

print('=' * 60)
print('ENSEMBLE INFERENCE (3 seeds rank fusion; spec = diagnostic only)')
print('=' * 60)

output_dir = Path(CFG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# Part 0: 定位成员 checkpoint
# ============================================================

def _iter_pt_files(root, max_depth=2):
    """限深度扫描 best_model_s*.pt — 避免 rglob 遍历竞赛数据集的数万目录。"""
    stack = [(root, 0)]
    while stack:
        d, depth = stack.pop()
        try:
            entries = list(d.iterdir())
        except OSError:
            continue
        for p in entries:
            try:
                if p.is_file() and p.name.startswith('best_model_s') and p.name.endswith('.pt'):
                    yield p
                elif p.is_dir() and depth < max_depth:
                    stack.append((p, depth + 1))
            except OSError:
                continue


def find_checkpoints():
    """扫描 ckpt_input 数据集 (+ Kaggle 全 input 兜底), 返回 [(key, path)]。

    key: int seed (s42/s142/s242) 或字符串 'spec' (词表专家, 按文件名识别)。
    seed 升序在前, spec 最后。
    """
    roots = [Path(CFG['ckpt_input'])]
    kg_input = Path('/kaggle/input')
    if kg_input.exists():  # Kaggle 环境: 兜底扫描全部挂载数据集
        roots += [p for p in kg_input.iterdir() if p.is_dir()]

    found = {}
    for root in roots:
        if not root.exists():
            continue
        for fp in sorted(_iter_pt_files(root, max_depth=2)):
            fp_str = str(fp)
            if fp_str in found.values():
                continue
            if fp.name == 'best_model_spec.pt':
                key = 'spec'
            else:
                m = re.match(r'best_model_s(\d+)\.pt$', fp.name)
                if not m:
                    continue
                key = int(m.group(1))
            try:
                ck = torch.load(fp_str, map_location='cpu', weights_only=False)
                # 配置一致性: 文件名的 seed 必须与 checkpoint 内 config 一致
                cfg_seed = (ck.get('config') or {}).get('seed')
                if key != 'spec' and cfg_seed != key:
                    print(f'  skip {fp_str}: filename seed {key} != config seed {cfg_seed}')
                    continue
                found[key] = fp_str
                print(f'  found: {key} <- {fp_str}')
            except Exception as e:
                print(f'  skip {fp_str}: {type(e).__name__}')
    order = sorted(k for k in found if k != 'spec') + \
            (['spec'] if 'spec' in found else [])
    return [(k, found[k]) for k in order]

checkpoints = find_checkpoints()
if not checkpoints:
    raise FileNotFoundError(
        f'未找到任何 best_model_s*.pt。请把训练产物 '
        f'(results/v5s{{1,2,3}}/checkpoints/best_model_s{{42,142,242}}.pt, '
        f'可选 results/v5spec/checkpoints/best_model_spec.pt) '
        f'上传为 Kaggle Dataset 并挂载到本 notebook (CFG["ckpt_input"])。')
print(f'\nMembers: {len(checkpoints)} | keys: {[k for k, _ in checkpoints]}')

# ============================================================
# Part 1: 配置交叉核对 (训练侧 vs 推理侧, 不一致立即停)
# ============================================================

# 数据侧参数必须与推理配置逐项一致; 训练侧独有参数 (lr/batch 等) 不检查
CHECK_KEYS = ['image_size', 'crop_mm', 'cache_slices', 'group_size',
              'center_pct', 'dinov2_variant', 'cls_dim', 'slot_hidden',
              'num_classes', 'unfreeze_layers', 'tta_jitter']

def member_label(key):
    """成员显示名/文件名后缀: int seed → s42, 'spec' → spec。"""
    return f's{key}' if key != 'spec' else 'spec'


ckpt_meta = {}
print('\n--- Checkpoint 交叉核对 ---')
for key, path in checkpoints:
    ck = torch.load(path, map_location='cpu', weights_only=False)
    cfg_ck = ck.get('config') or {}
    mismatches = [k for k in CHECK_KEYS
                  if k in cfg_ck and cfg_ck[k] != CFG.get(k)]
    if ck.get('targets') != TARGET_COLUMNS:
        mismatches.append('targets')
    if ck.get('slots') != SLOTS:
        mismatches.append('slots')
    if mismatches:
        raise ValueError(
            f'{member_label(key)} ({path}) 与推理配置不一致: {mismatches}。\n'
            f'  checkpoint 侧: ' + ' '.join(
                f'{k}={cfg_ck.get(k)}' for k in mismatches if k in cfg_ck) +
            f'\n  推理配置侧:    ' + ' '.join(
                f'{k}={CFG.get(k)}' for k in mismatches if k in CFG) +
            '\n  → 该 checkpoint 不属于本 v5 管线 (288px/130mm/jitter)，'
            '请检查上传的权重。')
    ema_ok = bool(ck.get('ema') and ck['ema'].get('shadow'))
    ckpt_meta[key] = ck
    print(f'  {member_label(key):>5s}: epoch={ck.get("epoch")}, AUC={ck.get("auc", 0):.4f}, '
          f'EMA={"OK" if ema_ok else "MISSING"}')

# ============================================================
# Part 2: 构建推理模型 (与训练 cell 17 同款: 权重全部来自 checkpoint)
# ============================================================

print('\nBuilding inference model (DINOv2-small @ 288px)...')
infer_backbone = timm.create_model(
    CFG['dinov2_variant'], pretrained=False, num_classes=0,
    img_size=CFG['image_size'])

infer_model = MultiViewModel(
    dinov2_model=infer_backbone, n_slots=N_SLOT, cls_dim=CFG['cls_dim'],
    n_classes=CFG['num_classes'], slot_hidden=CFG['slot_hidden'],
    dropout=0.0, unfreeze_layers=CFG['unfreeze_layers'],
).to(DEVICE)
infer_model.eval()


def load_member_weights(ck):
    """加载成员权重 (处理 DataParallel 前缀 + EMA shadow), 返回打印信息。"""
    state_dict = ck['model']
    if next(iter(state_dict)).startswith('module.'):
        state_dict = {k.replace('module.', '', 1): v for k, v in state_dict.items()}
    if ck.get('ema') and ck['ema'].get('shadow'):
        for name in state_dict:
            if name in ck['ema']['shadow']:
                state_dict[name] = ck['ema']['shadow'][name]
        ema_note = 'EMA'
    else:
        ema_note = 'raw'
    infer_model.load_state_dict(state_dict, strict=False)
    return ema_note


# ============================================================
# Part 3: gold 验证推理 (7 窗口 TTA + jitter + 诊断池化)
# ============================================================

print('\n--- Gold Validation (58 studies) ---')

gold_rows = []
for uid in gold_studies:
    ri = gold_study_index[uid]
    slots_all = torch.from_numpy(GOLD_CACHE[ri].copy())  # [6, 9, H, W]
    m = torch.from_numpy(GOLD_MASK[ri].copy())            # [6]
    windows = torch.stack(
        [slots_all[:, w:w + CFG['group_size']] for w in range(N_WINDOWS)], dim=0)
    gold_rows.append((windows, m, uid))

@torch.no_grad()
def infer_gold_batch(windows_batch, mask_batch, model):
    """TTA + 诊断池化（jitter 视图平均 → per-target 窗口池化）"""
    B = windows_batch.shape[0]
    W = N_WINDOWS
    flat = windows_batch.reshape(B * W, *windows_batch.shape[2:]).to(DEVICE)  # B-major
    flat_mask = mask_batch.unsqueeze(1).expand(B, W, -1).reshape(B * W, -1).to(DEVICE)
    if CFG.get('tta_jitter', False):
        flat = torch.cat([flat, tta_jitter(flat)], dim=0)   # [2*B*W, ...] 原始块在前
        flat_mask = flat_mask.repeat(2, 1)
        n_orig = W
    else:
        n_orig = None
    logits = model(flat, flat_mask)  # [V*B*W, 12]
    logits_v = stack_views(logits, B, W, n_orig)  # [B, V*W, 12]
    return diagnostic_pool(logits_v.cpu(), n_orig=n_orig)  # [B, 12]


def run_gold_inference(model):
    """全 58 gold 推理 → [58, 12] probs。"""
    probs_list, uids_list = [], []
    for start in range(0, len(gold_rows), 8):
        batch = gold_rows[start:start + 8]
        windows_batch = torch.stack([r[0] for r in batch])
        mask_batch = torch.stack([r[1] for r in batch])
        uids = [r[2] for r in batch]
        probs = infer_gold_batch(windows_batch, mask_batch, model)
        probs_list.append(probs)
        uids_list.extend(uids)
    return torch.cat(probs_list).numpy(), uids_list


# gold 真值标签
gold_labels_arr = np.zeros((len(gold_studies), N_CLASSES))
for i, uid in enumerate(gold_studies):
    for j, c in enumerate(TARGET_COLUMNS):
        raw = gold_labels.loc[uid, c] if uid in gold_labels.index else np.nan
        gold_labels_arr[i, j] = float(raw) if not pd.isna(raw) else 0.0


def compute_gold_aucs(probs):
    aucs = {}
    for i, c in enumerate(TARGET_COLUMNS):
        yt, yp = gold_labels_arr[:, i], probs[:, i]
        n_pos = int(yt.sum())
        if n_pos > 0 and n_pos < len(yt):
            try:
                aucs[c] = float(roc_auc_score(yt, yp))
            except Exception:
                aucs[c] = float('nan')
        else:
            aucs[c] = float('nan')
    valid = [v for v in aucs.values() if not math.isnan(v)]
    macro = float(np.mean(valid)) if valid else float('nan')
    return aucs, macro


# ============================================================
# Part 4: test 集 slot 匹配 + 缓存 (训练 cell 17 同款)
# ============================================================

print('\n--- Test Set Slot Matching ---')

test_df = pd.read_csv(comp_input / 'test.csv')
test_df['StudyInstanceUID'] = test_df['StudyInstanceUID'].astype(str)
test_dicom_root = comp_input / 'test_series'


def _find_dicom_files(series_dir):
    """列出目录中的 DICOM 文件（不依赖扩展名，竞赛 test 集 DICOM 无 .dcm 后缀）。"""
    all_files = sorted([f for f in series_dir.iterdir() if f.is_file()])
    dcm = [f for f in all_files if f.suffix == '.dcm']
    return dcm if dcm else [f for f in all_files if not f.name.startswith('.')]


def _scan_test_dicoms(dicom_root):
    """扫描测试集 DICOM 目录，从 header 推断 plane / fluid / fatsat。"""
    rows = []
    root = Path(dicom_root)
    if not root.exists():
        return rows
    for study_dir in sorted(root.iterdir()):
        if not study_dir.is_dir():
            continue
        study_uid = study_dir.name
        for series_dir in sorted(study_dir.iterdir()):
            if not series_dir.is_dir():
                continue
            series_uid = series_dir.name
            dcm_files = _find_dicom_files(series_dir)
            if not dcm_files:
                continue
            try:
                ds = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True, force=True)

                iop = getattr(ds, 'ImageOrientationPatient', None)
                plane = 'Axial'
                if iop is not None and len(iop) >= 6:
                    try:
                        row_cos = np.array([float(iop[0]), float(iop[1]), float(iop[2])])
                        col_cos = np.array([float(iop[3]), float(iop[4]), float(iop[5])])
                        normal = np.cross(row_cos, col_cos)
                        dominant = int(np.argmax(np.abs(normal)))
                        plane = {0: 'Sagittal', 1: 'Coronal', 2: 'Axial'}[dominant]
                    except Exception:
                        pass

                desc = str(getattr(ds, 'SeriesDescription', '')).lower()
                seq_name = str(getattr(ds, 'SequenceName', '')).lower()
                scan_opts = str(getattr(ds, 'ScanOptions', '')).upper()

                fs_kw = ['fs', 'fatsat', 'fat sat', 'stir', 'spair', 'spir', 'we',
                         'water excit', 'tirm', 'fatsup']
                has_fs = any(kw in desc for kw in fs_kw)
                has_fs = has_fs or any(kw in scan_opts for kw in ['FS', 'FATSAT', 'SPAIR', 'SPIR'])

                t1_kw = ['t1', 't1w']
                is_t1 = any(kw in desc or kw in seq_name for kw in t1_kw)
                is_t2 = any(kw in desc or kw in seq_name for kw in ['t2', 't2w'])
                is_pd = any(kw in desc for kw in ['pd', 'pdw', 'proton', 'dp', 'dens'])
                has_fluid = (is_t2 or is_pd) and not is_t1

                rows.append({
                    'StudyInstanceUID': study_uid,
                    'SeriesInstanceUID': series_uid,
                    'Anatomical_Plane': plane,
                    'Fluid_Sensitive': 1 if has_fluid else 0,
                    'Fat_Suppression': 1 if has_fs else 0,
                })
            except Exception:
                continue
    return rows


test_slot_map = {}
test_series_path = comp_input / 'test_series.csv'

if test_series_path.exists():
    test_series = pd.read_csv(test_series_path)
    test_series['StudyInstanceUID'] = test_series['StudyInstanceUID'].astype(str)
    test_series['SeriesInstanceUID'] = test_series['SeriesInstanceUID'].astype(str)
    print(f'test_series.csv: {len(test_series)} series, '
          f'{test_series["StudyInstanceUID"].nunique()} studies')

    test_slot_map, _ = build_study_slot_map(test_series, test_dicom_root)
    csv_studies = len(test_slot_map)

    if csv_studies < max(10, len(test_df) * 0.5):
        print(f'CSV coverage ({csv_studies}/{len(test_df)}) insufficient, '
              f'scanning DICOM headers...')
        dicom_rows = _scan_test_dicoms(test_dicom_root)
        if dicom_rows:
            test_series = pd.DataFrame(dicom_rows)
            test_slot_map, _ = build_study_slot_map(test_series, test_dicom_root)
            print(f'DICOM scan: {len(test_series)} series, '
                  f'{test_series["StudyInstanceUID"].nunique()} studies → '
                  f'{len(test_slot_map)} studies matched')
        else:
            print(f'DICOM scan returned 0 rows, keeping CSV results ({csv_studies} studies)')
else:
    print('test_series.csv not found, scanning DICOM headers...')
    dicom_rows = _scan_test_dicoms(test_dicom_root)
    if dicom_rows:
        test_series = pd.DataFrame(dicom_rows)
        test_slot_map, _ = build_study_slot_map(test_series, test_dicom_root)
        print(f'DICOM scan: {len(test_series)} series, '
              f'{len(test_slot_map)} studies matched')

test_studies = sorted(test_slot_map.keys())
print(f'Test studies with slot match: {len(test_studies)}/{len(test_df)}')

# ---- 测试缓存 (gold 缓存很小, 无需释放) ----
TEST_CACHE, TEST_MASK, test_study_idx = None, None, {}
if len(test_studies) > 0:
    n_test = len(test_studies)
    TEST_CACHE = np.zeros((n_test, N_SLOT, CFG['cache_slices'],
                           CFG['image_size'], CFG['image_size']), dtype=np.uint8)
    TEST_MASK = np.zeros((n_test, N_SLOT), dtype=np.float32)

    for row_idx, study_uid in enumerate(test_studies):
        test_study_idx[study_uid] = row_idx

    t0 = time.time()
    jobs = []
    for row_idx, study_uid in enumerate(test_studies):
        study_slots = test_slot_map[study_uid]
        for slot_idx, (slot_name, plane, fluid, fatsat) in enumerate(SLOTS):
            slot_info = study_slots.get(slot_name)
            if slot_info is not None:
                jobs.append((row_idx, slot_idx, slot_name, plane, slot_info, None))

    print(f'Decoding {len(jobs)} test slot-series...')
    completed, failed = 0, 0
    with ThreadPoolExecutor(max_workers=CFG['pix_threads']) as pool:
        for row_idx, slot_idx, result in pool.map(_read_slot_job, jobs):
            completed += 1
            if result is not None:
                TEST_CACHE[row_idx, slot_idx] = result
                TEST_MASK[row_idx, slot_idx] = 1.0
            else:
                failed += 1
            if completed % 1000 == 0:
                print(f'  [{completed}/{len(jobs)}] {time.time()-t0:.0f}s', flush=True)

    print(f'Test cache: {n_test} studies in {time.time()-t0:.0f}s ({failed} failed)')


@torch.no_grad()
def infer_test_batch(indices, model):
    windows_list, masks_list, empty_mask = [], [], []
    for idx in indices:
        windows_list.append(torch.stack([
            torch.from_numpy(TEST_CACHE[idx, :, w:w + CFG['group_size']].copy())
            for w in range(N_WINDOWS)
        ], dim=0))
        masks_list.append(torch.from_numpy(TEST_MASK[idx].copy()))
        empty_mask.append(TEST_MASK[idx].sum() == 0)

    windows_batch = torch.stack(windows_list)
    mask_batch = torch.stack(masks_list)

    B, W = windows_batch.shape[0], N_WINDOWS
    flat = windows_batch.reshape(B * W, *windows_batch.shape[2:]).to(DEVICE)  # B-major
    flat_mask = mask_batch.unsqueeze(1).expand(B, W, -1).reshape(B * W, -1).to(DEVICE)
    if CFG.get('tta_jitter', False):
        flat = torch.cat([flat, tta_jitter(flat)], dim=0)   # [2*B*W, ...] 原始块在前
        flat_mask = flat_mask.repeat(2, 1)
        n_orig = W
    else:
        n_orig = None
    logits = model(flat, flat_mask)
    probs = diagnostic_pool(
        stack_views(logits, B, W, n_orig).cpu(), n_orig=n_orig)  # [B, C]

    # Studies with no slots → fill 0.5
    for i, is_empty in enumerate(empty_mask):
        if is_empty:
            probs[i] = 0.5
    return probs


def run_test_inference(model):
    """全 test 推理 → [n_test, 12] probs (或 None)。"""
    if TEST_CACHE is None:
        return None
    n_test = TEST_CACHE.shape[0]
    test_probs = np.zeros((n_test, N_CLASSES), dtype=np.float32)
    t1 = time.time()
    for start in range(0, n_test, 8):
        idx = list(range(start, min(start + 8, n_test)))
        test_probs[idx] = infer_test_batch(idx, model).numpy()
        if start % 200 == 0:
            print(f'  [{start}/{n_test}] {time.time()-t1:.0f}s', flush=True)
    return test_probs

# ============================================================
# Part 5: 成员循环 — 逐成员 gold + test 推理
# ============================================================

print('\n--- Member inference ---')
member_gold_probs = {}   # key -> [58, 12]
member_test_probs = {}   # key -> [n_test, 12] (或 None)
member_aucs = {}         # key -> (per-class dict, macro)

for key, path in checkpoints:
    ck = ckpt_meta[key]
    tag = member_label(key)
    print(f'\n[{tag}] Loading {Path(path).name} ...')
    ema_note = load_member_weights(ck)
    print(f'  epoch={ck.get("epoch")}, AUC={ck.get("auc", 0):.4f}, weights={ema_note}')

    t0 = time.time()
    gold_probs, gold_uids = run_gold_inference(infer_model)
    aucs, macro = compute_gold_aucs(gold_probs)
    member_gold_probs[key] = gold_probs
    member_aucs[key] = (aucs, macro)
    print(f'  Gold ({len(gold_uids)} studies, {N_WINDOWS}-window TTA+jitter + diag pool): '
          f'Macro AUC {macro:.4f} ({time.time()-t0:.0f}s)')

    # 保存成员 gold 预测 (与训练产物同格式, 可逐位对比)
    gold_rows_out = []
    for i, uid in enumerate(gold_uids):
        row = {'StudyInstanceUID': uid}
        for j, c in enumerate(TARGET_COLUMNS):
            row[f'true_{c}'] = int(gold_labels_arr[i, j])
            row[f'prob_{c}'] = float(gold_probs[i, j])
        gold_rows_out.append(row)
    pd.DataFrame(gold_rows_out).to_csv(
        output_dir / f'gold_validation_predictions_{tag}.csv', index=False)
    auc_rows = [{'class': c, 'auc': aucs[c], 'n_pos': int(gold_labels_arr[:, i].sum())}
                for i, c in enumerate(TARGET_COLUMNS)]
    pd.DataFrame(auc_rows + [{'class': 'macro_avg', 'auc': macro, 'n_pos': 0}]
                 ).to_csv(output_dir / f'gold_validation_auc_{tag}.csv', index=False)

    if TEST_CACHE is not None:
        test_probs = run_test_inference(infer_model)
        member_test_probs[key] = test_probs
        print(f'  Test inference: {len(test_probs)} studies')

# ============================================================
# Part 6: rank-mean 融合 + 提交
# ============================================================

print('\n--- Fusion ---')

seeds = sorted(k for k in member_gold_probs if k != 'spec')
has_spec = 'spec' in member_gold_probs
if not has_spec:
    print('spec member not found — pure rank-mean over seed members')

# ★ spec 融合已归档 (2026-08-15 离线扫描 scripts/fusion_scan_spec.py, 58 gold):
#   S2 当前 4 类 0.75/0.75 替换 = -0.0023 (MCL -0.0295 拖累)
#   S3 仅 ACL 替换 = +0.0004 (w 任意 — rank 混合整体缩放不改变排序)
#   S4 spec 作第 4 成员 (最佳 w=0.15) = +0.0003
#   S5 逐类 oracle 上限 = +0.0011 —— 连作弊上限都低于 +0.003 阈值
#   → 蒸馏失败 (spec 宏 0.8606), 融合端禁用; 扫到 spec 文件只打印诊断 AUC
SPEC_REPLACE_TARGETS = []
SPEC_W = 0.75


def rank_mean(probs_list):
    """逐类 rankdata 平均 (rank 空间融合, 与 0.91 方案同构)。"""
    acc = np.zeros_like(probs_list[0], dtype=np.float64)
    for p in probs_list:
        r = rankdata(p, axis=0, method='average') / len(p)
        acc += r
    return acc / len(probs_list)


def rank_of(p):
    return rankdata(p, axis=0, method='average') / len(p)


def ensemble_fused(probs_by_member):
    """返回 (final, base): base = seed 成员 rank-mean;
    final = spec 在词表强类上按 0.75+0.75 重投票后的结果。"""
    if len(seeds) > 1:
        base = rank_mean([probs_by_member[s] for s in seeds])
    else:
        base = rank_of(probs_by_member[seeds[0]]) if seeds else None
    if base is None:
        return None, None
    if not has_spec:
        return base, base
    spec_rank = rank_of(probs_by_member['spec'])
    final = base.copy()
    for c in SPEC_REPLACE_TARGETS:
        j = TARGET_COLUMNS.index(c)
        final[:, j] = SPEC_W * base[:, j] + SPEC_W * spec_rank[:, j]
    return final, base


# gold 融合
fused_gold, base_gold = ensemble_fused(member_gold_probs)
print(f'gold: {len(seeds)} seed members rank-mean'
      + ('; spec member found but archived (not fused)' if has_spec else ''))

aucs_f, macro_f = compute_gold_aucs(fused_gold)
print(f'\n{"member":<12s} {"Macro AUC":>10s}')
for s in seeds:
    print(f's{s:<11d} {member_aucs[s][1]:10.4f}')
if has_spec:
    print(f'{"spec":<12s} {member_aucs["spec"][1]:10.4f}')
print(f'{"BASE":<12s} {compute_gold_aucs(base_gold)[1]:10.4f}'
      + ('  (纯 seed rank-mean)' if has_spec else ''))
print(f'{"FUSED":<12s} {macro_f:10.4f}'
      + ('  (= BASE, spec 已归档不参与)' if has_spec else ''))
if has_spec and SPEC_REPLACE_TARGETS:
    print(f'\n  替换类 per-class (base → fused):')
    base_aucs, _ = compute_gold_aucs(base_gold)
    for c in SPEC_REPLACE_TARGETS:
        print(f'    {c:<18s} {base_aucs[c]:.3f} → {aucs_f[c]:.3f} '
              f'({aucs_f[c] - base_aucs[c]:+.3f})')
print(f'\n★ 对照: 各成员读数应与各自训练会话一致 '
      f'(s42≈0.8933 / s142≈0.8899 / s242≈0.8937)。差 >0.005 说明产物错配。'
      + ('\n  spec 只作诊断 (蒸馏上限对照), 融合端已归档不参与。'
         if has_spec else ''))

# 保存融合 gold 预测 + AUC
def gold_rows_of(probs):
    rows = []
    for i, uid in enumerate(gold_studies):
        row = {'StudyInstanceUID': uid}
        for j, c in enumerate(TARGET_COLUMNS):
            row[f'true_{c}'] = int(gold_labels_arr[i, j])
            row[f'prob_{c}'] = float(probs[i, j])
        rows.append(row)
    return rows


pd.DataFrame(gold_rows_of(fused_gold)).to_csv(
    output_dir / 'gold_validation_predictions_fused.csv', index=False)
auc_rows_f = [{'class': c, 'auc': aucs_f[c], 'n_pos': int(gold_labels_arr[:, i].sum())}
              for i, c in enumerate(TARGET_COLUMNS)]
pd.DataFrame(auc_rows_f + [{'class': 'macro_avg', 'auc': macro_f, 'n_pos': 0}]
             ).to_csv(output_dir / 'gold_validation_auc_fused.csv', index=False)
if has_spec and SPEC_REPLACE_TARGETS:
    pd.DataFrame(gold_rows_of(base_gold)).to_csv(
        output_dir / 'gold_validation_predictions_fused_base.csv', index=False)

# ---- test 融合 → submission ----
submission_rows = []
if TEST_CACHE is not None:
    fused_test, _ = ensemble_fused(member_test_probs)
    for row_idx, study_uid in enumerate(test_studies):
        row = {'StudyInstanceUID': study_uid}
        for j, c in enumerate(TARGET_COLUMNS):
            row[c] = float(fused_test[row_idx, j])
        submission_rows.append(row)
else:
    print('No test DICOMs found — filling all studies with 0.5')

submission_df = pd.DataFrame(submission_rows)
full_submission = test_df[['StudyInstanceUID']].merge(
    submission_df, on='StudyInstanceUID', how='left')
for c in TARGET_COLUMNS:
    full_submission[c] = full_submission[c].fillna(0.5)

submission_path = output_dir / 'submission.csv'
full_submission.to_csv(submission_path, index=False)
print(f'\nSubmission saved: {submission_path}')
print(f'  Studies: {len(full_submission)} (expected: {len(test_df)})')
print(f'  Mean prob: {full_submission[TARGET_COLUMNS].values.mean():.4f}')
for c in TARGET_COLUMNS:
    vals = full_submission[c].values
    print(f'  {c:<20s}: mean={vals.mean():.4f}, std={vals.std():.4f}, '
          f'>0.5={np.mean(vals > 0.5):.1%}')

print(f'\nDone!')
print(f'  Members: {len(seeds)} seeds'
      + (' + spec found (archived, not fused)' if has_spec else '')
      + f' | Fused Gold AUC: {macro_f:.4f}')
print(f'  Submission: {submission_path}')
print(f'  Ready to submit to Kaggle!')

